# CBDB SQLite Setup

This notebook downloads the latest CBDB SQLite database and applies optional post-processing steps:

| Step | What it does |
|------|--------------|
| **Add Foreign Keys** | Reads `foreign_keys_regen.csv` from GitHub and recreates tables with proper `FOREIGN KEY` constraints |
| **Create Views** | Creates 18 convenience SQL views (e.g. `View_PeopleData`, `View_EntryData`) |
| **Create Addresses** | Builds the `ADDRESSES` hierarchy table from `ADDR_CODES` / `ADDR_BELONGS_DATA` |

**Usage:** Edit the *Configuration* cell below, then click **Runtime → Run all**.

## 0 · Setup
Clone the CBDB SQLite repository to make the helper scripts available.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/cbdb-project/cbdb_sqlite.git"
REPO_DIR = "/content/cbdb_sqlite"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

SCRIPTS_DIR = os.path.join(REPO_DIR, "scripts")
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

print(f"Repository ready at {REPO_DIR}")

## 1 · Configuration
Set each flag to `True` or `False` to control which steps run.

In [ ]:
# ── What to run ────────────────────────────────────────────────────────────────
ADD_FOREIGN_KEYS = True   # add FK constraints from foreign_keys_regen.csv
CREATE_VIEWS     = True   # create 18 convenience SQL views
CREATE_ADDRESSES = True   # build ADDRESSES hierarchy table
DOWNLOAD_RESULT  = True   # download the processed .db file (Colab only)

# ── Output path ────────────────────────────────────────────────────────────────
DB_PATH = "/content/cbdb_latest.db"

# ── Source URLs (no need to change) ────────────────────────────────────────────
LATEST_JSON_URL = (
    "https://raw.githubusercontent.com/cbdb-project/cbdb_sqlite/master/latest.json"
)
FK_CSV_URL = (
    "https://raw.githubusercontent.com/cbdb-project/cbdb-user-mdb-tests"
    "/main/reports/foreign_keys_regen.csv"
)

## 2 · Download & Extract Database

In [ ]:
import json, urllib.request, zipfile, shutil, os

print("Fetching latest.json...")
with urllib.request.urlopen(LATEST_JSON_URL) as r:
    info = json.loads(r.read().decode())

print(f"  File      : {info['sqlite_filename']}")
print(f"  Generated : {info['generated_at_utc']}")
print(f"  URL       : {info['huggingface_url']}")

if os.path.exists(DB_PATH):
    print(f"\nDatabase already exists at {DB_PATH}. Delete it to re-download.")
else:
    zip_path = "/content/_cbdb_download.zip"
    print("\nDownloading...")

    def _progress(count, block, total):
        if total > 0:
            pct = min(count * block * 100 / total, 100)
            print(f"\r  {pct:5.1f}%", end="", flush=True)

    urllib.request.urlretrieve(info["huggingface_url"], zip_path, _progress)
    print()  # newline after progress bar

    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as z:
        db_members = [
            m for m in z.namelist()
            if m.lower().endswith((".db", ".sqlite3", ".sqlite"))
        ]
        if not db_members:
            raise FileNotFoundError("No database file found in the downloaded archive.")
        z.extract(db_members[0], "/content/")
        shutil.move(f"/content/{db_members[0]}", DB_PATH)
    os.remove(zip_path)

    size_mb = os.path.getsize(DB_PATH) / 1024 / 1024
    print(f"Saved to {DB_PATH}  ({size_mb:.1f} MB)")

## 3 · Add Foreign Keys
Fetches `foreign_keys_regen.csv` from GitHub, parses the FK definitions,
and recreates each table that is missing its `FOREIGN KEY` constraints.
Tables that already have FK constraints are skipped (idempotent).

In [ ]:
if ADD_FOREIGN_KEYS:
    import add_foreign_keys
    add_foreign_keys.add_foreign_keys(DB_PATH, FK_CSV_URL)
else:
    print("Skipped (ADD_FOREIGN_KEYS = False).")

## 4 · Create Views
Runs `create_views.sh` to add 18 convenience SQL views such as
`View_PeopleData`, `View_EntryData`, `View_PostingOfficeData`, etc.

In [ ]:
if CREATE_VIEWS:
    import subprocess, os
    subprocess.run(["apt-get", "install", "-y", "-q", "sqlite3"], check=True)
    script = os.path.join(REPO_DIR, "scripts", "create_views.sh")
    result = subprocess.run(
        ["bash", script, DB_PATH],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("create_views.sh failed — see output above.")
    lines = result.stdout.splitlines()
    created = sum(1 for l in lines if l.startswith("Creating view"))
    print(f"Created {created} views.")
    for line in lines:
        if any(kw in line for kw in ("ERROR", "✗", "All sanity checks")):
            print(line)
else:
    print("Skipped (CREATE_VIEWS = False).")

## 5 · Create Addresses Table
Builds the `ADDRESSES` table by resolving the full administrative hierarchy
for each address across time, preserving gaps in the data.

In [ ]:
if CREATE_ADDRESSES:
    from create_addresses_table import AddressHierarchyBuilder
    with AddressHierarchyBuilder(DB_PATH) as builder:
        builder.run()
else:
    print("Skipped (CREATE_ADDRESSES = False).")

## 6 · Download Result
Triggers a browser download of the processed database.
Only works inside Google Colab; outside Colab the file path is printed instead.

In [ ]:
if DOWNLOAD_RESULT:
    try:
        from google.colab import files
        print(f"Starting download of {DB_PATH}...")
        files.download(DB_PATH)
    except ImportError:
        print(f"Not running in Colab. Your database is at: {DB_PATH}")
else:
    print("Skipped (DOWNLOAD_RESULT = False).")